# Building an EVALS

In [1]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math

from pathlib import Path

from utils.schema_json import ReportData

from pydantic import BaseModel

Added to sys.path: C:\Users\lucat\PythonRepositories\PRIN


In [2]:
DEVELOPER_PROMPT_FILE_NAME = "system_prompt_3_openai.txt"

TEST_DATA_FILE_NAME = "data_luca_test.jsonl"

In [3]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

MODEL = 'gpt-4o-mini'

In [4]:
#Carica developer prompt da file
developer_prompt_path = Path('../models/prompts/' + DEVELOPER_PROMPT_FILE_NAME)

with open(developer_prompt_path, "r", encoding="utf-8") as f:
    DEVELOPER_PROMPT = f.read()

print(DEVELOPER_PROMPT)

# Load training data from file
test_data_path = Path('../data/ft-dataset/' + TEST_DATA_FILE_NAME)

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo in uno schema JSON.


In [6]:
class ReportText(BaseModel):
    report_text: str
    
display(ReportText.model_json_schema())

test_report = ReportText(report_text=test_data[0]['messages'][1]['content'])
print(test_report.report_text)

{'properties': {'report_text': {'title': 'Report Text', 'type': 'string'}},
 'required': ['report_text'],
 'title': 'ReportText',
 'type': 'object'}

IN CORRISPONDENZA DEL RETTO MEDIO-ALTO , DA CIRCA 6 CM DALLO SFINTERE ANALE INTERNO CON ESTENSIONE CRANIALE PER CIRCA 5 CM SI SEGNALA ISPESSIMENTO PARIETALE CIRCONFERENZIALE CHE INVIA DIGITAZIONI POLIPOIDI NEL MESORETTO CHE INGLOBANO I VASI EMORROIDARI E RETRAE IN UN PUNTO IL PERITONEO.
UN ALTRO DEPOSITO TUMORALE CHE INGLOBA I VASI EMORROIDARI È PRESENTE PIU' CRANIALMENTE LUNGO I VASI EMORROIDARI SUPERIORI.
LINFONODI SOSPETTI NEL MESORETTO IN NUMERO SUPERIORE A TRE , UNO DEI QUALI ADESO ALLA FASCIA PERIRETTALE ALLE ORE 8 , ED IN SEDE LATERALE A SINISTRA, I MAGGIORI CON ASSE CORTO DI CIRCA 7 MM.
ALCUNI DIVERTICOLI IN CORRISPONDENZA DEL SIGMA .
CONCLUSIONI: STADIAZIONE RM T4AN2 MRF+ EXTRAMESORETTALE  EMVI


In [57]:
def extract_data_from_report(report_text: str):
    response = client.responses.parse(
        model=MODEL,
        input=[
            {'role': 'developer', 'content': DEVELOPER_PROMPT},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=ReportData
    )
    return response

In [58]:
result = extract_data_from_report(test_report.report_text)
if False:
    print(result.output_text)

In [59]:
if True:
    print(result.output_text)

{"morfologia":"solido_polipoide","posizione":"medio","spessore_parietale":null,"estensione_cranio_caudale":5,"distanza_oai":6,"riflessione_peritoneale_anteriore":"sotto","infiltrazione_tessuto_adiposo":"sospetto","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","coinvolgimento_riflessione_peritoneale":"si","coinvolgimento_fascia_mesorettale":"si","linfonodi_sospetti":4,"sedi_locoregionali":{"mesorettali":true,"rettali_superiori":false,"mesenterici_inferiori":false,"iliaci_interni":false,"otturatori":false,"sacrali":false,"inguinali_sotto_dentata":false},"sedi_non_locoregionali":{"inguinali":false,"iliaci_esterni":false,"iliaci_comuni":false,"paraortici":false,"altri":false},"depositi_tumorali":"si","emvi_esteso":"si","carcinosi_peritoneale":"no","lesioni_ossee":"no"}


In [63]:
result.output_parsed.model_dump(mode='json')

{'morfologia': 'solido_polipoide',
 'posizione': 'medio',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 5,
 'distanza_oai': 6,
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'sospetto',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'coinvolgimento_riflessione_peritoneale': 'si',
 'coinvolgimento_fascia_mesorettale': 'si',
 'linfonodi_sospetti': 4,
 'sedi_locoregionali': {'mesorettali': True,
  'rettali_superiori': False,
  'mesenterici_inferiori': False,
  'iliaci_interni': False,
  'otturatori': False,
  'sacrali': False,
  'inguinali_sotto_dentata': False},
 'sedi_non_locoregionali': {'inguinali': False,
  'iliaci_esterni': False,
  'iliaci_comuni': False,
  'paraortici': False,
  'altri': False},
 'depositi_tumorali': 'si',
 'emvi_esteso': 'si',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no'}

In [ ]:
# We want our input data to be available in our variables, so we set the item_schema to
# ReportText.model_json_schema()
data_source_config = {
    "type": "custom",
    "item_schema": ReportText.model_json_schema(),
    # We're going to be uploading completions from the API, so we tell the Eval to expect this
    "include_sample_schema": True,
}

{
    "properties": {
        "report_text": {
            "title": "Report Text",
            "type": "string"
        }
    },
    "required": [
        "report_text"
    ],
    "title": "ReportText",
    "type": "object"
}


The item schema

In [39]:
print(json.dumps(data_source_config['item_schema'], indent=4))

{
    "properties": {
        "report_text": {
            "title": "Report Text",
            "type": "string"
        }
    },
    "required": [
        "report_text"
    ],
    "title": "ReportText",
    "type": "object"
}


Means that we'll have the variable `{{item.report_text}}` available in our eval.

`"include_sample_schema": True` Mean's that we'll have the variable `{{sample.output_text}}` available in our eval.

In [7]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": developer_prompt},
        {
            "role": "user",
            "content": test_report,
        },
    ],
    text_format=ReportData,
)

report_data = response.output_parsed

In [10]:
report_data.model_dump(mode='json')

{'morfologia': 'solido_anulare',
 'posizione': 'alto',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 5,
 'distanza_oai': 110,
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'si_5mm',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'no',
 'linfonodi_sospetti': 16,
 'sedi_locoregionali': {'mesorettali': True,
  'rettali_superiori': True,
  'mesenterici_inferiori': False,
  'iliaci_interni': True,
  'otturatori': True,
  'sacrali': False,
  'inguinali_sotto_dentata': False},
 'sedi_non_locoregionali': {'inguinali': False,
  'iliaci_esterni': True,
  'iliaci_comuni': False,
  'paraortici': False,
  'altri': False},
 'depositi_tumorali': 'no',
 'emvi_esteso': 'no',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no'}